In [ ]:
!pip install -q pydantic ollama

In [ ]:
import json
import ollama
import re
from typing import Literal, TypeAlias, TypeVar
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
try:
    prompt_urgencia = open("prompt_urgencia.md", "r", encoding="utf-8").read()
    prompt_categoria = open("prompt_categoria.md", "r", encoding="utf-8").read()
    prompt_accion = open("prompt_accion.md", "r", encoding="utf-8").read()
    prompt_respuesta = open("prompt_respuesta.md", "r", encoding="utf-8").read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

In [ ]:
T = TypeVar("T", bound=BaseModel)

TipoUrgencia: TypeAlias = Literal["Alta", "Baja"]
TipoCategoria: TypeAlias = Literal["Envío/Logística", "Facturación", "Producto Defectuoso"]
TipoPolitica: TypeAlias = Literal["Generar ticket", "Reembolsar", "Pedir más datos", "Reprogramar envío"]
TipoCodigoAccion: TypeAlias = Literal["ACC_GENERAR_TICKET", "ACC_REEMBOLSO", "ACC_PEDIR_DATOS", "ACC_REPROGRAMAR_ENVIO"]

class SalidaUrgencia(BaseModel):
    nivel_urgencia: TipoUrgencia

class SalidaCategoria(BaseModel):
    categoria: TipoCategoria

class SalidaAccion(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: TipoPolitica
    codigo_accion_sistema: TipoCodigoAccion

class SalidaRespuesta(BaseModel):
    mensaje_al_cliente: str

class Metadatos(BaseModel):
    categoria: TipoCategoria
    nivel_urgencia: TipoUrgencia

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: TipoPolitica

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: TipoCodigoAccion
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

In [ ]:
class OutputValidator:
    def __init__(self):
        self.suspicious_patterns = [
            r'##\s*ROL\s+Sos\s+un',
            r'##\s*(INSTRUCCI[OÓ]N|REGLAS DE SEGURIDAD|TONO|VALORES PERMITIDOS|EJEMPLOS|INPUT|OUTPUT)',
            r'API[_\s]KEY[:=]\s*\w+',
        ]

    def validate_output(self, output: str) -> bool:
        return not any(re.search(pattern, output, re.IGNORECASE)
                      for pattern in self.suspicious_patterns)

    def filter_response(self, response: str) -> str:
        if not self.validate_output(response) or len(response) > 1500:
            return "No puedo procesar solicitudes que entren en conflicto con mis directrices operativas."
        return response

In [ ]:
def llamar_llm(system_prompt: str, input_data: dict, esquema: type[T]) -> T:
    mensajes = [
        {
            "role": "system",
            "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido."
        },
        {
            "role": "user",
            "content": json.dumps(input_data, ensure_ascii=False)
        }
    ]
    respuesta = ollama.chat(
        model="qwen3:8b",
        messages=mensajes,
        format=esquema.model_json_schema(),
        options={"temperature": 0.1}
    )
    return esquema.model_validate_json(respuesta['message']['content'])

In [ ]:
def procesar_mensaje(mensaje_usuario: str) -> dict:

    validator = OutputValidator()

    print("Procesado urgencia...")
    out_urgencia = llamar_llm(prompt_urgencia, {"mensaje_usuario": mensaje_usuario}, SalidaUrgencia)
    nivel_urgencia = out_urgencia.nivel_urgencia

    print("Procesado categoria...")
    out_categoria = llamar_llm(prompt_categoria, {"mensaje_usuario": mensaje_usuario}, SalidaCategoria)
    categoria = out_categoria.categoria

    print("Procesado accion...")
    out_accion = llamar_llm(
        prompt_accion,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria},
        SalidaAccion
    )
    cadena_de_pensamiento = out_accion.cadena_de_pensamiento
    politica_aplicada = out_accion.politica_aplicada
    codigo_accion_sistema = out_accion.codigo_accion_sistema

    print("Procesado respuesta...")
    out_respuesta = llamar_llm(
        prompt_respuesta,
        {"mensaje_usuario": mensaje_usuario, "nivel_urgencia": nivel_urgencia, "categoria": categoria, "politica_aplicada": politica_aplicada},
        SalidaRespuesta
    )
    mensaje_al_cliente = validator.filter_response(out_respuesta.mensaje_al_cliente)

    salida_validada = SalidaMaestra(
        metadatos_clasificacion=Metadatos(
            categoria=categoria,
            nivel_urgencia=nivel_urgencia
        ),
        analisis_interno=Analisis(
            cadena_de_pensamiento=cadena_de_pensamiento,
            politica_aplicada=politica_aplicada
        ),
        accion_y_respuesta=AccionRespuesta(
            codigo_accion_sistema=codigo_accion_sistema,
            mensaje_al_cliente=mensaje_al_cliente
        )
    )

    return salida_validada.model_dump()